### Column Transformer

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute  import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [4]:
df = pd.read_csv("../../datasets/covid_toy.csv")

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    str    
 2   fever      90 non-null     float64
 3   cough      100 non-null    str    
 4   city       100 non-null    str    
 5   has_covid  100 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 4.8 KB


In [17]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [15]:
strings = df.select_dtypes(["str"])
for col in strings:
    print()
    print(df[col].value_counts())


gender
Female    59
Male      41
Name: count, dtype: int64

cough
Mild      62
Strong    38
Name: count, dtype: int64

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

has_covid
No     55
Yes    45
Name: count, dtype: int64


In [9]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [12]:
X_train, X_test, y_train , y_test = train_test_split(X,y,
                                                     test_size=0.2)

In [18]:
X_train

,age,gender,fever,cough,city
35,82,Female,102.0,Strong,Bangalore
65,69,Female,102.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
58,23,Male,98.0,Strong,Mumbai
32,34,Female,101.0,Strong,Delhi
...,...,...,...,...,...
28,16,Male,104.0,Mild,Kolkata
50,19,Male,101.0,Mild,Delhi
64,42,Male,104.0,Mild,Mumbai
53,83,Male,98.0,Mild,Delhi


#### Without using column transfromation

In [22]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.transform(X_test[["fever"]])
X_train_fever.shape

(80, 1)

In [26]:
oe = OrdinalEncoder(categories=[['Mild',"Strong"]])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [29]:
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[["gender","city"]])
X_test_gender_city = ohe.transform(X_test[["gender","city"]])
X_train_gender_city.shape

(80, 4)

In [33]:
X_train_age = X_train.drop(["gender","fever", "cough",'city'], axis=1).values
X_test_age = X_test.drop(["gender","fever", "cough",'city'], axis=1).values

X_train_age.shape


(80, 1)

In [38]:
X_train_trasform = np.concatenate((X_train_age,X_train_gender_city,X_train_fever,X_train_cough), axis=1)
X_test_trasform = np.concatenate((X_test_age,X_test_gender_city,X_test_fever,X_test_cough), axis=1)

X_train_trasform.shape

(80, 7)

#### using column transfromer

In [42]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),["fever"]),
    ('tnf2', OrdinalEncoder(categories=[["Mild","Strong"]]),["cough"]),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),["gender","city"])
    
    ], remainder='passthrough')

In [44]:
transformer.fit_transform(X_train).shape

(80, 7)

In [47]:
transformer.transform(X_test).shape

(20, 7)